# v58 — KTF³ Mirror Correlation: GPU-Optimized

**For Mateja Radojičić — 4x RTX 5090**

**Authors:** Andrew Cotting (KTF³) + Mateja Radojičić (Twin Barrier Theory)

## What this notebook does

Tests the KTF³ prediction P1: mirror cross-correlation peak at r = T₁/2 = 830 Mpc.

KTF³ identification: φ(x,y,z) = (−x,−y,−z)

For each galaxy at position p, its mirror is at -p.
The mirror cross-correlation C(r) should peak at r = 830 Mpc if KTF³ is correct.

## Data

BOSS DR12 galaxy catalog (public) — ~1.2 million galaxies, z = 0.2-0.7

## GPU optimization

- CuPy for GPU array operations (drop-in NumPy replacement)
- Parallel pair counting on GPU
- Expected speedup: 50-100x vs CPU
- On 4x RTX 5090: can process 10M pairs/second

## Pre-registration

OSF: osf.io/42nks (April 1, 2026)
GitHub: github.com/Andrew-Cot/KTF3-Analyse

In [ ]:
# Install packages
!pip install cupy-cuda12x numpy scipy matplotlib astropy -q 2>/dev/null || \
 pip install cupy numpy scipy matplotlib astropy -q

# Try to import CuPy (GPU), fall back to NumPy (CPU)
try:
    import cupy as cp
    GPU_AVAILABLE = True
    print(f'✅ GPU available: {cp.cuda.Device().name}')
    print(f'   Memory: {cp.cuda.Device().mem_info[1]/1e9:.1f} GB')
except ImportError:
    import numpy as cp
    GPU_AVAILABLE = False
    print('⚠️  No GPU — running on CPU (slower)')

import numpy as np
import matplotlib.pyplot as plt
from astropy.cosmology import FlatLambdaCDM
from astropy.io import fits
import os, time
print('Ready.')

In [ ]:
# === KTF³ PARAMETERS ===
T1_Mpc  = 1660.0   # Fundamental period
r_pred  = T1_Mpc / 2  # = 830 Mpc — mirror correlation peak
cosmo   = FlatLambdaCDM(H0=67.4, Om0=0.315)

# Bin settings
r_min, r_max, dr = 100, 1400, 50  # Mpc
bins = np.arange(r_min, r_max + dr, dr)
bin_centers = 0.5 * (bins[:-1] + bins[1:])
N_BINS = len(bin_centers)

# Monte Carlo settings
N_PAIRS = 5_000_000   # pairs per run (GPU can handle 50M+)
N_MC    = 500         # null simulations

print(f'KTF³ prediction: mirror peak at r = {r_pred} Mpc')
print(f'Bins: {r_min}-{r_max} Mpc, width={dr} Mpc')
print(f'Pairs per run: {N_PAIRS:,}')
print(f'GPU mode: {GPU_AVAILABLE}')

In [ ]:
# === DOWNLOAD BOSS DR12 ===
# Public data from SDSS
import urllib.request

BOSS_FILE = 'boss_dr12_galaxies.fits'

if not os.path.exists(BOSS_FILE):
    print('Downloading BOSS DR12...')
    # LOWZ+CMASS combined catalog
    urls = [
        'https://data.sdss.org/sas/dr12/boss/lss/galaxy_DR12v5_CMASSLOWZTOT_South.fits.gz',
        'https://data.sdss.org/sas/dr12/boss/lss/galaxy_DR12v5_CMASSLOWZTOT_North.fits.gz',
    ]
    for url in urls:
        fname = url.split('/')[-1]
        print(f'  Trying: {fname}')
        try:
            urllib.request.urlretrieve(url, fname)
            print(f'  Downloaded: {os.path.getsize(fname)/1e6:.0f} MB')
        except Exception as e:
            print(f'  Failed: {e}')
else:
    print(f'Already exists: {os.path.getsize(BOSS_FILE)/1e6:.0f} MB')

# If download fails, generate synthetic catalog for pipeline test
if not os.path.exists(BOSS_FILE):
    print('Using synthetic catalog (pipeline test only)')
    USE_SYNTHETIC = True
    N_GAL = 100_000
    np.random.seed(42)
    ra_all  = np.random.uniform(100, 260, N_GAL)
    dec_all = np.random.uniform(-10, 60, N_GAL)
    z_all   = np.random.uniform(0.2, 0.7, N_GAL)
else:
    USE_SYNTHETIC = False
    hdul = fits.open(BOSS_FILE)
    data = hdul[1].data
    ra_all  = np.array(data['RA'],  dtype=float)
    dec_all = np.array(data['DEC'], dtype=float)
    z_all   = np.array(data['Z'],   dtype=float)
    hdul.close()
    print(f'Loaded {len(ra_all):,} galaxies')

In [ ]:
# === CONVERT TO CARTESIAN ===
# Quality cuts
good = (z_all > 0.2) & (z_all < 0.7) & np.isfinite(ra_all) & np.isfinite(dec_all)
ra  = ra_all[good]
dec = dec_all[good]
z   = z_all[good]

print(f'Galaxies after cuts: {len(ra):,}')

# Comoving distance
chi = cosmo.comoving_distance(z).value  # Mpc

# Cartesian
ra_rad  = np.radians(ra)
dec_rad = np.radians(dec)
x = chi * np.cos(dec_rad) * np.cos(ra_rad)
y = chi * np.cos(dec_rad) * np.sin(ra_rad)
z_cart = chi * np.sin(dec_rad)

pos = np.stack([x, y, z_cart], axis=1).astype(np.float32)
print(f'Position array: {pos.shape}, dtype={pos.dtype}')
print(f'Distance range: {chi.min():.0f} - {chi.max():.0f} Mpc')

# KTF³ mirror: phi(x,y,z) = (-x,-y,-z)
# Mirror positions in galactic coordinates (l=277, b=-31)
# For simplicity: full inversion (valid for isotropic test)
pos_mirror = -pos  # phi(x,y,z) = (-x,-y,-z)

# Move to GPU if available
if GPU_AVAILABLE:
    pos_gpu        = cp.array(pos)
    pos_mirror_gpu = cp.array(pos_mirror)
    print('Arrays transferred to GPU')
else:
    pos_gpu        = pos
    pos_mirror_gpu = pos_mirror

In [ ]:
# === GPU MIRROR CORRELATION ===
def compute_correlation_gpu(pos, pos_ref, bins, n_pairs, rng_seed=42):
    """
    Compute cross-correlation between pos and pos_ref.
    Randomly samples n_pairs pairs for efficiency.
    Returns: bin counts normalized by n_pairs
    """
    N = len(pos)
    rng = np.random.default_rng(rng_seed)
    idx1 = rng.integers(0, N, n_pairs)
    idx2 = rng.integers(0, N, n_pairs)
    keep = idx1 != idx2
    idx1, idx2 = idx1[keep], idx2[keep]

    if GPU_AVAILABLE:
        idx1_gpu = cp.array(idx1)
        idx2_gpu = cp.array(idx2)
        dr = pos[idx1_gpu] - pos_ref[idx2_gpu]
        r  = cp.sqrt(cp.sum(dr**2, axis=1))
        r_cpu = cp.asnumpy(r)
    else:
        dr = pos[idx1] - pos_ref[idx2]
        r  = np.sqrt(np.sum(dr**2, axis=1))
        r_cpu = r

    counts, _ = np.histogram(r_cpu, bins=bins)
    return counts / len(idx1)

print('Computing mirror correlation...')
t0 = time.time()

# Standard correlation (galaxy-galaxy)
C_std    = compute_correlation_gpu(pos_gpu, pos_gpu, bins, N_PAIRS, seed=42)

# Mirror correlation (galaxy-mirror)
C_mirror = compute_correlation_gpu(pos_gpu, pos_mirror_gpu, bins, N_PAIRS, seed=43)

# Excess
C_excess = C_mirror - C_std

t1 = time.time()
print(f'Done in {t1-t0:.1f}s')

peak_bin = np.argmax(C_excess)
peak_r   = bin_centers[peak_bin]
print(f'Mirror excess peak at: r = {peak_r} Mpc')
print(f'Predicted peak at:     r = {r_pred} Mpc')

In [ ]:
# === MONTE CARLO NULL TEST ===
print(f'Running {N_MC} Monte Carlo simulations...')
t0 = time.time()

mc_excess = np.zeros((N_MC, N_BINS))

for i in range(N_MC):
    # Random mirror axis (null hypothesis)
    rng = np.random.default_rng(i + 1000)
    theta = rng.uniform(0, np.pi)
    phi   = rng.uniform(0, 2*np.pi)
    n = np.array([np.sin(theta)*np.cos(phi),
                  np.sin(theta)*np.sin(phi),
                  np.cos(theta)], dtype=np.float32)

    # Random mirror: reflect through plane perpendicular to n
    if GPU_AVAILABLE:
        n_gpu = cp.array(n)
        pos_rand_mirror = pos_gpu - 2 * cp.outer(pos_gpu @ n_gpu, n_gpu)
    else:
        pos_rand_mirror = pos - 2 * np.outer(pos @ n, n)

    C_rand = compute_correlation_gpu(pos_gpu, pos_rand_mirror, bins, N_PAIRS, seed=i)
    mc_excess[i] = C_rand - C_std

    if (i+1) % 100 == 0:
        elapsed = time.time() - t0
        print(f'  {i+1}/{N_MC}  ({elapsed:.0f}s elapsed)')

mc_mean = mc_excess.mean(axis=0)
mc_std  = mc_excess.std(axis=0)
sigma   = (C_excess - mc_mean) / (mc_std + 1e-12)

pred_bin   = np.argmin(np.abs(bin_centers - r_pred))
sigma_pred = sigma[pred_bin]

print(f'\nσ at r={r_pred} Mpc: {sigma_pred:.3f}')
print(f'Max σ overall: {sigma.max():.3f} at r={bin_centers[sigma.argmax()]} Mpc')

In [ ]:
# === PLOT ===
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('v58 — KTF³ Mirror Correlation (GPU) — Cotting & Radojičić (2026)',
             fontsize=12, fontweight='bold')

ax = axes[0]
ax.fill_between(bin_centers, mc_mean-2*mc_std, mc_mean+2*mc_std,
                alpha=0.3, color='gray', label='2σ null')
ax.plot(bin_centers, C_excess, 'r-', lw=2, label='Mirror excess')
ax.axvline(r_pred, color='blue', lw=2, ls='--',
           label=f'KTF³ prediction: {r_pred} Mpc')
ax.axhline(0, color='black', lw=0.8)
ax.set_xlabel('r [Mpc]')
ax.set_ylabel('C_mirror − C_std')
ax.set_title('Mirror Correlation Excess')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(bin_centers, sigma, 'r-', lw=2)
ax.axhline(0, color='black', lw=0.8)
ax.axhline(2, color='orange', lw=1.5, ls='--', label='2σ')
ax.axhline(-2, color='orange', lw=1.5, ls='--')
ax.axvline(r_pred, color='blue', lw=2, ls='--',
           label=f'Predicted: {r_pred} Mpc')
ax.set_xlabel('r [Mpc]')
ax.set_ylabel('Significance σ')
ax.set_title(f'Significance (σ at {r_pred} Mpc = {sigma_pred:.2f})')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('v58_mirror_correlation_GPU.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: v58_mirror_correlation_GPU.png')

In [ ]:
# === SUMMARY ===
print('=' * 60)
print('v58 — KTF³ Mirror Correlation (GPU-Optimized)')
print('Cotting, S. & Radojičić, M. (2026)')
print('=' * 60)
print(f'Data:        {"SYNTHETIC" if USE_SYNTHETIC else "BOSS DR12"}')
print(f'Galaxies:    {len(ra):,}')
print(f'GPU:         {GPU_AVAILABLE}')
print(f'N_pairs:     {N_PAIRS:,}')
print(f'N_MC:        {N_MC}')
print()
print(f'KTF³ prediction: r = {r_pred} Mpc')
print(f'σ at prediction: {sigma_pred:.3f}')
print(f'Max σ:           {sigma.max():.3f} at r={bin_centers[sigma.argmax()]:.0f} Mpc')
print()
if abs(sigma_pred) < 1:
    print('Verdict: NULL')
elif abs(sigma_pred) < 2:
    print('Verdict: MARGINAL')
else:
    print(f'Verdict: SIGNAL ({sigma_pred:.1f}σ)')
print()
print('OSF pre-registration: osf.io/42nks')
print('GitHub: github.com/Andrew-Cot/KTF3-Analyse')
print('=' * 60)